1. TF - Term Frequency

O tf mede quanto um termo aparece em um documento

Exemplo:

- this movie is amazing and this movie is emotional

Como da pra ver, "movie" aparece duas vezes, portanto o tf de movie é maior que o de uma palavra que aparece apenas uma vez.

Idealmente, quanto mais uma palavra aparece mais ela pode ser importante para descrever aquele documento.

2. IDF - Inverse Document Frequency

O IDF mede o quao rara uma palavra é no corpus inteiro.

Se uma palavra aparece em quase todos os documentos, ela nao ajuda muito a distinguir um documento dos outros.

- Movie aparece em milhares de reviews masterpiece aparece em poucas
- Manterpiece aparece em poucas

Entao:

- Movie -> IDF menor
- Masterpiece -> IDF maior

Essencialmente palavras raras tem maior poder discriminativo.

3. TF-IDF

- TF = Term Frequency
- IDF = Inverse Document Frequency

O peso final combina os dois:

- TF_IDF(t, d) = TF(t, d) X IDF(t)

Onde:

- t = termo
- d = documento

Essencialmente vai pegar o termo mais frequente do documento e que seja mais raro no corpus(nao aparece em tantos documentos) e atribui o peso mais alto.

A ideia central é dar um peso numérico para cada palavra em cada documento.

Entao se um termo foi muito comum em todos os documentos, o IDF dele vai ser menor.

Esse peso tenta responder:

- "Essa palavra é importante neste documento, mas nao tao comum no corpus inteiro?"

Porque palavras como:
- this, is, movie
Podem aparecer em muitos textos e, por isso, ajudam pouco a diferenciar documentos.

Já palavras mais específicas, como:
- masterpierce, boring, suspenseful

Podem carregar mais informação

TF-IDF foi por muito tempo uma das bases dos sistemas de recuperação porque ele resolve dois problemas:

1. não trata todas as palavras como igualmente importantes
2. permite transformar texto em vetor numérico

Depois disso, você pode:

1. comparar consulta e documentos
2. calcular similaridade do cosseno
3. ranquear resultados

In [10]:
import pandas as pd
#1. Carregar dataset

PATH = r"C:\Users\User\Documents\datasets\tabulate\IMDB Dataset.csv"

df = pd.read_csv(PATH)

print(df.head())
print(df.columns)
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
Index(['review', 'sentiment'], dtype='str')
(50000, 2)


In [11]:
#2. Pré-processamento simples
import re

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)          # remove HTML
    text = re.sub(r"[^a-z\s]", " ", text)       # mantém letras e espaços
    text = re.sub(r"\s+", " ", text).strip()    # remove espaços extras
    return text

df["clean_review"] = df["review"].apply(preprocess_text)

print(df[["review", "clean_review"]].head(3))

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   

                                        clean_review  
0  one of the other reviewers has mentioned that ...  
1  a wonderful little production the filming tech...  
2  i thought this was a wonderful way to spend ti...  


In [12]:
#3. Gerar a matriz TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X = vectorizer.fit_transform(df["clean_review"])

print("Formato da matriz TF-IDF:", X.shape)

Formato da matriz TF-IDF: (50000, 5000)


In [13]:
#4. Ver o vocabulário
feature_names = vectorizer.get_feature_names_out()

print(feature_names[:50])

['aaron' 'abandoned' 'abc' 'abilities' 'ability' 'able' 'absence' 'absent'
 'absolute' 'absolutely' 'absurd' 'abuse' 'abused' 'abusive' 'abysmal'
 'academy' 'accent' 'accents' 'accept' 'acceptable' 'accepted' 'access'
 'accident' 'accidentally' 'accompanied' 'accomplished' 'according'
 'account' 'accuracy' 'accurate' 'accurately' 'accused' 'achieve'
 'achieved' 'achievement' 'acid' 'act' 'acted' 'acting' 'action' 'actions'
 'active' 'activities' 'actor' 'actors' 'actress' 'actresses' 'acts'
 'actual' 'actually']


In [14]:
#5. Ver os pesos TF-IDF de um documento
doc_index = 0

doc_vector = X[doc_index].toarray()[0]

nonzero_indices = doc_vector.nonzero()[0]

tfidf_terms = [(feature_names[i], doc_vector[i]) for i in nonzero_indices]

tfidf_terms_sorted = sorted(tfidf_terms, key=lambda x: x[1], reverse=True)

print("Top termos TF-IDF do documento 0:\n")
for term, score in tfidf_terms_sorted[:20]:
    print(f"{term}: {score:.4f}")

Top termos TF-IDF do documento 0:

oz: 0.5635
violence: 0.2477
prison: 0.2279
forget: 0.1956
struck: 0.1758
ll: 0.1419
word: 0.1241
city: 0.1229
episode: 0.1210
high: 0.1020
christians: 0.0999
guards: 0.0983
agenda: 0.0963
experimental: 0.0953
away: 0.0946
painted: 0.0924
darker: 0.0920
hardcore: 0.0915
right: 0.0907
comfortable: 0.0907


In [15]:
#6. Fazer uma consulta com TF-IDF
query = ["amazing emotional movie"]

q_vec = vectorizer.transform(query)

print("Formato do vetor da consulta:", q_vec.shape)

Formato do vetor da consulta: (1, 5000)


In [16]:
#7. Comparar consulta com os documentos
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(q_vec, X).flatten()

print(scores[:10])

[0.         0.         0.         0.02193034 0.00739227 0.02273343
 0.         0.07418881 0.         0.04457889]


In [17]:
#8. Recuperar os top documentos
top_n = 5
top_indices = scores.argsort()[-top_n:][::-1]

print("Top resultados:\n")

for idx in top_indices:
    print(f"Documento {idx} | Score: {scores[idx]:.4f}")
    print("Sentiment:", df.iloc[idx]["sentiment"])
    print("Review:", df.iloc[idx]["review"][:300], "...\n")

Top resultados:

Documento 18331 | Score: 0.3256
Sentiment: positive
Review: Mani sir as usual brings out another amazing story with Kannathil Muthamittal. Such an amazing relationship between parents and child is brought out in a beautiful fashion. Mani Sir as usual without much special effects and not much outdoor shoots.(In fact this was the only movie where he went outsi ...

Documento 4388 | Score: 0.3221
Sentiment: positive
Review: I have seen this film three times now, and each time I see it, it becomes more personal and more emotional to watch.<br /><br />The acting is amazing, which is not hard to believe since it is Daniel Day Lewis, who is an amazing actor. Brenda Fricker is the surprise wonder in it, though. She captures ...

Documento 20774 | Score: 0.3193
Sentiment: positive
Review: In a word...amazing.<br /><br />I initially was not too keen to watch Pinjar since I thought this would be another movie lamenting over the partition and would show biases towards India and Pa

Como interpretar isso conceitualmente

Quando você usa TF-IDF, o processo é:

- cada review vira um vetor
- cada termo do vocabulário vira uma dimensão
- o valor em cada dimensão é o peso TF-IDF
- a consulta também vira um vetor

você compara consulta e documentos com similaridade do cosseno

Então o TF-IDF não é só “contagem de palavras”.

Ele é uma forma de representar documentos em um espaço vetorial, destacando termos mais informativos.

Quando TF-IDF é bom?

TF-IDF funciona bem quando:

- Você quer um baseline forte e simples
- o corpus é textual e relativamente limpo
- as consultas dependem muito de palavras exatas
- você quer algo rápido e interpretável

TF-IDF tem limitações importantes:

1. não entende semântica
2. não entende sinônimos
3. não entende contexto profundo
4. depende muito das palavras exatas

Exemplo:

- great film
- excellent movie

Esses textos são semanticamente parecidos, mas o TF-IDF pode não capturar isso bem se não houver sobreposição lexical.

Por isso ele foi muito usado historicamente, mas hoje costuma ser combinado com:

- BM25
- embeddings
- rerankers

Resumo mental útil:

- TF = “essa palavra é importante neste documento?”
- IDF = “essa palavra é rara no corpus?”
- TF-IDF = “essa palavra é importante neste documento e útil para diferenciá-lo?”